# Imports

In [ ]:
import cda2
#import datetime
import pyspark.sql.functions as F
import pyspark.sql.types as T
import json

from datetime import datetime, timedelta
from pyspark.sql.window import Window
from pyspark.sql.functions import col, row_number

# Connect to Spark

In [ ]:
api = cda2.Api()

Set configuration parameters to better optimize queries.

In [ ]:
config = {
    "spark.sql.adaptive.enabled": "true",
    "spark.sql.adaptive.coalescePartitions.enabled": "true",
    "spark.sql.adaptive.coalescePartitions.parallelismFirst": "false",
    "spark.sql.adaptive.coalescePartitions.minPartitionSize": "1m",
    "spark.executor.memory": "8g",
    "spark.executor.memoryOverhead": "16g",
}

Start Spark and specify number of cpus to use. 400 is quite high, but we'll be running 1 year at a time and want to have it done in just a few minutes.

In [ ]:
#api.start_spark(n_executors=100, config=config)
api.start_spark(n_executors=100)

Function to convert Unix timestamp (milliseconds from 1970) to YYYYMMDD string.

In [ ]:
# @F.udf("string")
# def to_date(ts):
#     return datetime.datetime.utcfromtimestamp(ts / 1000).strftime("%Y%m%d")

In [ ]:
%run ./shared_variables.ipynb

In [ ]:
#year0 = "2025"
year1 = str(int(year0) + 1)

In [ ]:
dates = {"start_date": year0 + "-01-01", "end_date": year1 +"-01-01"}

In [ ]:
print("retrieving fixes: ", datetime.now())

In [ ]:
# df_fixes_raw = (
#     api.dataframe("ArincFix", **dates, metadata=True)
#     .withColumn("uniq_fix_name", F.concat("identification.name", F.lit("("), "identification.icao_region", F.lit(")")))
#     .select(
#         "uniq_fix_name",
#         F.col("identification.name").alias("fix_name"),
#         F.col("arinc_record_info.customer_area_code").alias("area_code"),
#         F.col("identification.icao_region").alias("icao_region"),
#         "latitude",
#         "longitude",
#         "is_waypoint",
#         F.col("waypoint_info.type").alias("waypoint_type"),
#         F.col("waypoint_info.usage").alias("waypoint_usage"),
#         F.col("waypoint_info.name_format").alias("waypoint_name_format"),
#         F.col("waypoint_info.full_name").alias("waypoint_full_name"),
#         "is_vhf_navaid",
#         "is_ndb_navaid",
#         F.col("navaid_info.clazz").alias("navaid_class"),
#         F.col("navaid_info.facility_name").alias("navaid_facility_name"),
#         F.col("navaid_info.dme_latitude").alias("dme_latitude"),
#         F.col("navaid_info.dme_longitude").alias("dme_longitude"),
#         F.col("magnetic_variation.modeled").alias("magnetic_variation"),
#         F.col("metadata.effective_end_date").alias("end_date"),
#     )
#     .orderBy("uniq_fix_name")
# )

# #df_fixes_raw.show()

In [ ]:
df_fixes_raw = (
    api.dataframe("ArincFix", **dates, metadata=True)
    .withColumn("fix_name_uniq", F.concat("identification.name", F.lit("("), "identification.icao_region", F.lit(")")))
    .select(
        "fix_name_uniq",
        F.col("identification.name").alias("fix_name"),
        F.col("arinc_record_info.customer_area_code").alias("area_code"),
        F.col("identification.icao_region").alias("icao_region"),
        "latitude",
        "longitude",
        F.col("navaid_info.dme_latitude").alias("dme_latitude"),
        F.col("navaid_info.dme_longitude").alias("dme_longitude"),
        F.col("magnetic_variation.modeled").alias("magnetic_variation"),
        F.col("metadata.effective_end_date").alias("end_date"),
    )
    .orderBy("fix_name_uniq")
)

#df_fixes_raw.show()

In [ ]:
#df_fixes_raw.count()

In [ ]:
#df_fixes_raw.show()

In [ ]:
#print("   count starting: ", datetime.now())

In [ ]:
window = Window.partitionBy("fix_name_uniq").orderBy(col("end_date").desc())

df_fixes = (df_fixes_raw
    .withColumn("row", row_number().over(window))
    .filter(col("row") == 1)
    .drop("row")
    .drop("end_date")
)

#df_fixes.show()

In [ ]:
#df_fixes.count()

In [ ]:
#print("   count completed: ", datetime.now())

In [ ]:
#df_fixes.show()

In [ ]:
(
    df_fixes
    .repartition(1)
    .write.option("header", True)
    .csv("CRAFT/" + year0 + "/fixes", compression="None", mode="overwrite")
)

In [ ]:
#print("completed fixes: ", datetime.now())